## deblurring
repairing images motion blur

- structural re-parameterization (RepVGG): 3x3 + 1x1 + identity branches while training, fused into a single 3x3 convolution for inference [[paper]](https://arxiv.org/abs/2101.03697)
- residual blocks with parameter-free attention (f * tanh(f)), architecture from [dlss5-distill](https://github.com/Benjamin-Wegener/dlss5-distill) (Apache-2.0)
- global input bypass: the network only learns the residual (the missing sharpness)
- running on [pytorch](https://pytorch.org/) / [google colab](https://colab.research.google.com/), cuda, apple silicon (mps) or cpu
- trained from scratch (no pretrained weights)
- using the famous [Set14](https://www.google.com/search?q=set14) dataset ONLY (with heavy augmentation) - no validation needed

pytorch counterpart of [deblurring.ipynb](deblurring.ipynb)

jupyter notebook by [Benjamin Wegener](https://scholar.google.de/citations?user=yEn9St8AAAAJ) from [github](https://www.github.com/Benjamin-Wegener/keras-examples)

### options

In [ ]:
run_on_google_colab = True #change this to 'False' to run locally
epochs = 250 #Number of epochs to train
channels = 3 #channels of low resolution image
batch_size = 14 #what batch-size should we use (decrease if you encounter video memory errors)
steps_per_epoch = 1000 #How much iterations per epoch to train
height_lr = 256 #height of low resolution image
width_lr = height_lr #width of low resolution image
gen_lr = 0.001 #learning rate of generator
logging_steps = 50 #how often to update the training log
rotation_max = 0.33 # max max degree of rotation
max_move = height_lr / 100 #how much pixels to shift the blurred image
blur_copies = 5 # how many copies of image to generate and merge back together

### imports

In [ ]:
import os
if run_on_google_colab:
    %cd /content
    !git clone https://github.com/Benjamin-Wegener/keras-examples #download Dataset
    %cd keras-examples
import glob
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.ndimage import rotate, shift
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import clear_output
%matplotlib inline
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print('using device:', device)

### function for image visualization

In [ ]:
def show(tensors):
    plt.rcParams['figure.figsize'] = [20, 10]
    fig = plt.figure()
    for i in range(len(tensors)):
        t = tensors[i].detach().cpu().numpy() if torch.is_tensor(tensors[i]) else tensors[i]
        try:
            t = np.squeeze(t, axis = 0)
        except:
            pass
        t = np.transpose(t, (1, 2, 0)) #channels first -> channels last
        fig.add_subplot(1,len(tensors), i + 1)
        plt.imshow((t.clip(0, 1) * 255).astype(np.uint8), interpolation = 'nearest')
    plt.setp(plt.gcf().get_axes(), xticks=[], yticks=[]);
    plt.show()

### dataset function

In [ ]:
# load the Set14 images once, values in [0, 1]
images = [np.array(Image.open(f).convert('RGB')).astype(np.float32) / 255. for f in sorted(glob.glob('./Set14/*.png'))]
print(len(images), 'images found')

# return n_samples augmented target images (height x width), channels first
def get_hr_batch(n_samples, height, width):
    batch = np.empty((n_samples, height, width, channels), dtype = np.float32)
    for i in range(n_samples):
        img = random.choice(images)
        #random zoom (crop between 20% and 70% of the image), flips, rotations and brightness
        zoom = random.uniform(0.2, 0.7)
        h, w = max(int(img.shape[0] * zoom), 16), max(int(img.shape[1] * zoom), 16)
        y0, x0 = random.randint(0, img.shape[0] - h), random.randint(0, img.shape[1] - w)
        crop = Image.fromarray((img[y0:y0 + h, x0:x0 + w] * 255).astype(np.uint8)).resize((width, height), Image.BICUBIC)
        crop = np.asarray(crop).astype(np.float32) / 255.
        if random.random() < 0.5: crop = crop[:, ::-1]
        if random.random() < 0.5: crop = crop[::-1]
        crop = np.rot90(crop, random.randint(0, 3)) if height == width else crop
        batch[i] = (crop * random.uniform(0.5, 1.5)).clip(0, 1)
    return torch.from_numpy(batch).permute(0, 3, 1, 2)

# source images are blurred (slightly rotated, moved and merged together), x is a numpy image in [0, 255]
# (linear interpolation, order=1, is ~20x faster than the 5th order splines of the keras notebook and plenty for a blur)
def motion_blur(x, max_move):
    copy = x.copy()
    counter = 1
    for j in range(blur_copies):
        if (random.random() * 2 > 1) or (j == 1): #50% chance to make more than one copy
            rotation_angle = random.random() * rotation_max / 2 + rotation_max #minimum ratation is the half of maximum rotation
            copy = rotate(copy, rotation_angle, reshape=False, order=1, mode='reflect')
            move_amount_x = int(random.random() * max_move + 1) * 2 #allow for negative values
            move_amount_x = move_amount_x - move_amount_x / 2
            move_amount_y = int(random.random() * max_move + 1) * 2
            move_amount_y = move_amount_y - move_amount_y / 2
            copy = shift(copy, [move_amount_x, move_amount_y, 0], order=1, mode='reflect')
            counter = counter + 1
            x = (1 - 1 / counter) * x + 1 / counter * copy #blend back together
    return np.clip(x, 0, 255)

# return batch of augmented train and target images with quantity n_samples
def get_batch(n_samples, height, width, channels):
    y_train = get_hr_batch(n_samples, height, width)
    x_train = np.empty(tuple(y_train.shape), dtype = np.float32)
    for i in range(n_samples):
        blurred = motion_blur(y_train[i].permute(1, 2, 0).numpy() * 255., max_move)
        x_train[i] = np.transpose(blurred / 255., (2, 0, 1))
    return torch.from_numpy(x_train).to(device), y_train.to(device)

### base functions

In [ ]:
class RepConv3x3(nn.Module): #3x3 + 1x1 + identity branch while training, one single 3x3 convolution after switch_to_deploy()
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.in_ch, self.out_ch = in_ch, out_ch
        self.dense = nn.Conv2d(in_ch, out_ch, 3, 1, 1)
        self.point = nn.Conv2d(in_ch, out_ch, 1, 1, 0)
        self.identity = in_ch == out_ch
        self.act = nn.PReLU(out_ch)
        self.fused = None

    def forward(self, x):
        if self.fused is not None:
            return self.act(self.fused(x))
        out = self.dense(x) + self.point(x)
        if self.identity:
            out = out + x
        return self.act(out)

    def switch_to_deploy(self):
        if self.fused is not None:
            return
        k = self.dense.weight.data + F.pad(self.point.weight.data, [1, 1, 1, 1])
        b = self.dense.bias.data + self.point.bias.data
        if self.identity:
            for i in range(self.in_ch):
                k[i, i, 1, 1] += 1.0
        self.fused = nn.Conv2d(self.in_ch, self.out_ch, 3, 1, 1).to(k.device)
        self.fused.weight.data = k
        self.fused.bias.data = b
        del self.dense, self.point

class ResidualBlock(nn.Module): #two RepConvs and parameter-free attention
    def __init__(self, ch):
        super().__init__()
        self.conv1 = RepConv3x3(ch, ch)
        self.conv2 = RepConv3x3(ch, ch)

    def forward(self, x):
        feat = self.conv2(self.conv1(x))
        return x + feat * torch.tanh(feat)

class Generator(nn.Module):
    def __init__(self, channels = 3, base_ch = 28, num_blocks = 4, scale = 1):
        super().__init__()
        self.scale = scale
        self.head = RepConv3x3(channels, base_ch)
        self.blocks = nn.ModuleList([ResidualBlock(base_ch) for _ in range(num_blocks)])
        self.trunk = RepConv3x3(base_ch, base_ch)
        if scale == 2: #subpixel convolution / pixelshuffle
            self.up = nn.Sequential(nn.Conv2d(base_ch, base_ch * 4, 3, 1, 1), nn.PixelShuffle(2), nn.PReLU(base_ch), nn.Conv2d(base_ch, channels, 3, 1, 1))
        else:
            self.up = nn.Sequential(nn.Conv2d(base_ch, base_ch, 3, 1, 1), nn.PReLU(base_ch), nn.Conv2d(base_ch, channels, 3, 1, 1))
        nn.init.normal_(self.up[-1].weight, std = 1e-3) #start almost as the plain bypass (tiny residual) - keeps training from scratch stable,
        nn.init.zeros_(self.up[-1].bias) #but not exactly zero: pixels sitting exactly on the clamp limits would get no gradient at all
        self.skip_scale = nn.Parameter(torch.ones(1, channels, 1, 1))
        self.skip_bias = nn.Parameter(torch.zeros(1, channels, 1, 1))
        self.res_gain = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, x):
        #global bypass (bicubic for 2x, the input itself for 1x), the network only learns the residual
        bypass = F.interpolate(x, scale_factor = 2, mode = 'bicubic', align_corners = False) if self.scale == 2 else x
        bypass = bypass * self.skip_scale + self.skip_bias
        f0 = self.head(x)
        feat = f0
        for block in self.blocks:
            feat = block(feat)
        res = self.up(f0 + self.trunk(feat)) * self.res_gain
        return torch.clamp(bypass + res, 0., 1.)

    def switch_to_deploy(self):
        for m in self.modules():
            if isinstance(m, RepConv3x3):
                m.switch_to_deploy()

### build generator model

In [ ]:
generator = Generator(channels = channels, scale = 1).to(device)
print('parameters while training:', sum(p.numel() for p in generator.parameters()))

### train

In [ ]:
#load checkpoint & create the optimizer
print('trying to load last saved weights...', end = ' ')
try:
    generator.load_state_dict(torch.load('deblurring_dlss_distill_weights.pt', map_location = 'cpu'))
    print('success.')
except:
    print('failed')
    pass

optimizer = torch.optim.Adam(generator.parameters(), lr = gen_lr)

# Train generator
generator.train()
for real_epoch in range(epochs):
    for step in range(steps_per_epoch):
        X, Y = get_batch(batch_size, height_lr, width_lr, channels)
        loss = F.l1_loss(generator(X), Y) #mae, like in the keras notebooks
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step % logging_steps == 0:
            testX, testY = get_batch(1, height_lr, width_lr, channels)
            clear_output()
            print('epoch', real_epoch + 1, '/', epochs, '--> step', step, '/', steps_per_epoch, ': loss', loss.item())
            with torch.no_grad():
                testZ = generator(testX)
            show([testX, testZ, testY])
            print('test_loss:', F.l1_loss(testZ, testY).item())
    try:
        print('trying to save weights...', end = ' ')
        torch.save(generator.state_dict(), 'deblurring_dlss_distill_weights.pt')
        print('success.')
    except:
        print('failed.')

### validate on complete picture

In [ ]:
# show image in actual size https://stackoverflow.com/a/42314798/
def display_image_in_actual_size(im_data):
    dpi = 100
    height, width, depth = im_data.shape
    figsize = width / float(dpi), height / float(dpi)
    fig = plt.figure(figsize=figsize)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.axis('off')
    ax.imshow(im_data, cmap='gray')
    plt.show()

In [ ]:
psnr = lambda a, b: (10 * torch.log10(1. / F.mse_loss(a, b))).item()

testY = np.array(Image.open('./Set14/lenna.png').convert('RGB')).astype(np.float32)
testX = motion_blur(testY, testY.shape[0] / 100) / 255.
testX, testY = [torch.from_numpy(np.transpose(a, (2, 0, 1)))[None].float().to(device) for a in (testX, testY / 255.)]

# fuse the training branches into single 3x3 convolutions - the result stays the same
generator.eval()
with torch.no_grad():
    before = generator(testX)
generator.switch_to_deploy()
with torch.no_grad():
    predicted = generator(testX)
print('max difference after fusing the branches:', (before - predicted).abs().max().item())
print('psnr input/bicubic: %.2f dB, psnr result: %.2f dB' % (psnr(testX, testY), psnr(predicted, testY)))
show([testX, predicted, testY])
predicted = Image.fromarray((np.transpose(np.squeeze(predicted.cpu().numpy()), (1, 2, 0)).clip(0, 1) * 255).astype(np.uint8))

print('trying to save image as \'deblurring_dlss_distill_result.png\'...', end = ' ')
try:
    predicted.save('deblurring_dlss_distill_result.png', "PNG")
    print('success.')
except:
    print('failed.')
    pass